In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import pandas as pd
from itertools import product

# Read the dataframe
df = pd.read_csv("/Users/surya/Desktop/riverline_takehome/dev/conversation_bundle_flat.csv")


In [ ]:
import sys
sys.path.append("..")

from visualise_conv import show_conversation

an3_samples= ['003d61b9-9929-0d01-78c0-72d65befe630', '007fd69e-f588-87d7-edef-6847b0ff176e']
an2_samples= ['022f42f7-2654-206e-c2e5-100b86c215ee', '03f02884-3f00-12b6-2300-93e089e3fec6']
an1_samples= ['0398595b-e02f-56a1-73e9-3788e6aba235', '04fe76b0-23de-3a1b-d545-915eec901fbc']
an123_samples=  ['00ab8e48-5dd9-3bb5-c054-216ab58c9fda',
 '0116e0f9-0373-5d3a-149d-7b18d3c3c579',
 '01d2bc92-a9d8-1c8c-3a16-3bc390f1de16',
 '0261f428-dcd8-21c5-227c-f8d0502cc8d2',
 '032101dc-a78c-9aa9-c8e2-5f92b42f1dfa',
 '051670b7-b772-dc86-433a-099a40b8ab18',
 '05739a43-4de4-af02-6b9c-81092e72bcff',
 '081f905f-2867-2616-d97a-37ee7ebcf7db',
 '0d279dd8-65e4-98e1-549d-f47998bba681',
 '0fdf1e6b-da43-39e8-de9e-5e1e45207173']

# sample = an123_samples[3]
sample = "a17b15b2-9027-296a-0819-a86daa1bdeb4"
# Example: show a conversation (replace conversation_id as needed)
show_conversation(sample)

In [ ]:
- What types of spec violations are most common
- Which violations correlate with bad outcomes (complaints, regulatory flags)
- Statistical analysis: violation rates by borrower segment (language, DPD, temperament)
- Specific conversation examples with evidence 



### generating eval vx

In [ ]:
# ── Generate eval JSONL (versioned) ─────────────────────────────────────────
# Run AgentEvaluator on all production logs and save to dev/eval_v<N>.jsonl
# Increment EVAL_VERSION each time you want a fresh snapshot.

import json
import sys
from pathlib import Path

DEV_DIR = Path(".").resolve()
PS_DIR  = DEV_DIR.parent / "problem_statement"
sys.path.insert(0, str(PS_DIR))

from eval_takehome import AgentEvaluator  # noqa: E402

EVAL_VERSION = 2                                     # bump for each new run
EVAL_OUT = DEV_DIR / f"eval_v{EVAL_VERSION}.jsonl"

def _iter_jsonl(path: Path):
    with open(path, encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                yield json.loads(line)

evaluator = AgentEvaluator()
n_written = 0
with open(EVAL_OUT, "w", encoding="utf-8") as out_f:
    for conv in _iter_jsonl(PS_DIR / "data" / "production_logs.jsonl"):
        result = evaluator.evaluate(conv)
        record = {
            "conversation_id": conv["conversation_id"],
            "quality_score":   result["quality_score"],
            "risk_score":      result["risk_score"],
            "violations":      result["violations"],
        }
        out_f.write(json.dumps(record, ensure_ascii=False) + "\n")
        n_written += 1

print(f"Wrote {n_written} records → {EVAL_OUT}")


In [ ]:
### spec violations count 

import collections

# Read the eval file and parse violations
violation_counter = collections.Counter()
violation_name_map = {}
violation_name2conv_id_list = {}

with open(EVAL_OUT, encoding='utf-8') as f:
    for line in f:
        if not line.strip():
            continue
        rec = json.loads(line)
        for v in rec.get("violations", []):
            rule_id = v.get("rule")
            violation_counter[rule_id] += 1
            # Get the expanded name if available (assuming explanation gives name)
            violation_name_map[rule_id] = v.get("explanation", "")
            violation_name2conv_id_list[rule_id] = violation_name2conv_id_list.get(rule_id, []) + [rec["conversation_id"]]

total_violations = sum(violation_counter.values())
print(f"Total number of violations: {total_violations}\n")

print("Frequency of violation types:")
for rule, freq in violation_counter.most_common():
    name = violation_name_map.get(rule, "")
    print(f"  {rule}: {freq}  ->  {name[:90].replace('\n',' ')}{'...' if len(name) > 90 else ''}")

# If you want a dict mapping violation type to their name (shortened)
violation_type_to_name = {rule: (name[:90] + ('...' if len(name) > 90 else '')) for rule, name in violation_name_map.items()}

In [ ]:
import sys
sys.path.append("..")

from visualise_conv import show_conversation

an3_samples= ['003d61b9-9929-0d01-78c0-72d65befe630', '007fd69e-f588-87d7-edef-6847b0ff176e']
an2_samples= ['022f42f7-2654-206e-c2e5-100b86c215ee', '03f02884-3f00-12b6-2300-93e089e3fec6']
an1_samples= ['0398595b-e02f-56a1-73e9-3788e6aba235', '04fe76b0-23de-3a1b-d545-915eec901fbc']
an123_samples=  ['00ab8e48-5dd9-3bb5-c054-216ab58c9fda',
 '0116e0f9-0373-5d3a-149d-7b18d3c3c579',
 '01d2bc92-a9d8-1c8c-3a16-3bc390f1de16',
 '0261f428-dcd8-21c5-227c-f8d0502cc8d2',
 '032101dc-a78c-9aa9-c8e2-5f92b42f1dfa',
 '051670b7-b772-dc86-433a-099a40b8ab18',
 '05739a43-4de4-af02-6b9c-81092e72bcff',
 '081f905f-2867-2616-d97a-37ee7ebcf7db',
 '0d279dd8-65e4-98e1-549d-f47998bba681',
 '0fdf1e6b-da43-39e8-de9e-5e1e45207173']

sample = an123_samples[3]
# Example: show a conversation (replace conversation_id as needed)
show_conversation(sample)

## Analysis Summary\n
\n
The violation-outcome correlation analysis reveals several critical insights:\n
\n
### 🔴 High-Risk Violations\n
**State machine violations** are the strongest predictors of bad outcomes, increasing intervention risk by 74%. These include:\n
- `ACT_ZCM_TIMEOUT_INVALID_CONTEXT` - timeout actions in wrong states\n
- `INV_EXIT_STATE_NOT_FINAL` - conversations continue after escalation/dormancy\n
- `TR_INVALID_STATE_TRANSITION` - illegal state transitions\n
\n
**Compliance violations** like `CMP_DNC_VIOLATION` (continuing after \"don't contact\") increase intervention risk by 70% and have severe regulatory implications.\n
\n
### 🟢 Protective Violations\n
Surprisingly, some violations correlate with *better* outcomes:\n
- `AMT_SETTLEMENT_AMOUNT_OUT_OF_BOUNDS` reduces intervention risk by 48%\n
- Even \"wrong\" settlement offers indicate engagement vs. complete conversation breakdown\n
\n
### 📈 Business Impact\n
- **41% of conversations require human intervention** - very high baseline\n
- **11% complaint rate** - significant customer satisfaction issue\n
- **State machine reliability** is the #1 factor determining conversation success\n
\n
### 🎯 Recommendations\n
1. **Fix state machine logic** - highest impact on reducing interventions\n
2. **Enforce DNC compliance** - highest regulatory risk\n
3. **Improve settlement bounds** - high frequency but protective when working\n
4. **Quality issues are secondary** - repetition doesn't predict bad outcomes

In [ ]:
from violation_correlation_simple import load_eval_data, load_csv_data 

conversations_with_outcomes = load_csv_data('/Users/surya/Desktop/riverline_takehome/dev/conversation_bundle_flat.csv')
evaluations = load_eval_data('/Users/surya/Desktop/riverline_takehome/dev/eval_v2.jsonl')


flagged_outcome_conversations = [
    conv for conv in conversations_with_outcomes 
    if conv['complaint_flag'] or conv['regulatory_flag']
]

print(f"Found {len(flagged_outcome_conversations)} flagged out of {len(conversations_with_outcomes)} conversations")

In [ ]:

from violation_correlation_simple import * 
main()

In [ ]:
# Separate conversation IDs with bad outcomes into:
# (A) Those present in the annotations (evaluations), and
# (B) Those NOT present in the annotations.
# This allows easy filtering when reviewing evaluation coverage.

# First, build a set of annotated conversation_ids (those evaluated in the evals file)

bad_outcome_conversation_ids_by_flag = {}

for flag in ["complaint_flag", "regulatory_flag", "required_intervention"]:
    annotated = [
        conv["conversation_id"]
        for conv in conversations_with_outcomes
        if conv.get(flag) and conv.get("annotated")
    ]
    non_annotated = [
        conv["conversation_id"]
        for conv in conversations_with_outcomes
        if conv.get(flag) and not conv.get("annotated")
    ]
    bad_outcome_conversation_ids_by_flag[flag] = {
        "annotated": annotated,
        "non_annotated": non_annotated
    }



In [ ]:
for k, v in bad_outcome_conversation_ids_by_flag.items():
    print(f"{k}: {len(v['annotated'])} vs {len(v['non_annotated'])} conversations")
    print(v['annotated'][:3])



In [ ]:
from dev.visualise_conv import show_conversation
# sample = '018ddf34-d2f6-ae46-76fa-33c67ae24235'
# sample = bad_outcome_conversation_ids_by_flag['regulatory_flag']["annotated"][3]
sample = "a5f2726d-cfc6-f1dd-91ec-bd45a38d3af9"
sample = "9f655840-583e-dd5d-2894-b0b09456cab8"
sample = "d1a842f2-337c-0274-a4c1-a98b8b75709f"
sample = "a5f2726d-cfc6-f1dd-91ec-bd45a38d3af9"
sample = "2de90f78-3c50-8eae-1098-ffe5fbaf6888"
sample = "d1a842f2-337c-0274-a4c1-a98b8b75709f"
sample = "5e21f706-117e-4ef1-7e9b-7c3dc135123c"
sample = "4bba13dd-d738-8bf6-50e0-97de70ba0c66"
show_conversation(sample, eval_jsonl_file='/Users/surya/Desktop/riverline_takehome/dev/eval_v2.jsonl')
# show_conversation(sample)

In [ ]:
from violation_correlation_analysis import *

In [ ]:
correlation_df, score_analysis_df, merged_df = main()

## Segment-wise Violation Analysis

Runs the segment analysis script across `language`, `dpd_bucket`, `pos_bucket`,
`turn_length`, `behavioral`. Writes every table to CSV and both charts
(heatmap + outcome-lift bar) to PNG under `dev/segment_analysis_output/`.

Returned dict `bundle` has:
- `segment_summary[axis]`  — T1 per segment
- `rule_rates[axis]`       — T2 (rule rate + lift vs global)
- `stats[axis]`            — segment x rule x outcome with OR, risk diff, Fisher p
- `examples[axis]`         — 2 high-risk + 2 representative per top rule
- `top_findings`           — cross-segment strongest risk differences
- `examples_all`           — concatenated evidence registry

In [ ]:
from segment_violation_analysis import run_segment_analysis

bundle = run_segment_analysis()

In [ ]:
for axis in ["language", "dpd_bucket", "pos_bucket", "turn_length", "behavioral"]:
    print(f"\n====== {axis} ======")
    display(bundle["segment_summary"][axis])
    display(bundle["rule_rates"][axis].pivot(index=axis, columns="rule", values="lift").round(2))

In [ ]:
print("Top cross-segment findings (largest |risk diff| with support >= 15):")
display(bundle["top_findings"].head(15))

print("\nEvidence registry preview:")
display(bundle["examples_all"].head(12))

## Rule-level Cross-segment Profiles

For each violation class (`rule`), inspect how prevalence shifts across borrower segment values.

- Input: `bundle["rule_segment_matrix"]`
- Export: `dev/segment_analysis_output/rule_segment_matrix.csv`
- Charts: `dev/segment_analysis_output/charts/by_rule/<RULE>.png`

This is the **rule-first** view: one rule across all segment axes.

In [ ]:
matrix = bundle["rule_segment_matrix"].copy()

for rule, grp in matrix.groupby("rule"):
    g = grp.copy()
    g["rate_in_segment"] = g["rate_in_segment"].round(3)
    g["lift_vs_global"] = g["lift_vs_global"].round(2)

    print(f"\n=== {rule}  (global rate: {g['global_rate'].iloc[0]:.1%}) ===")

    pivot = g.pivot(index="segment_axis", columns="segment_value", values="lift_vs_global")
    display(pivot)

    display(
        g.sort_values(["segment_axis", "lift_vs_global"], ascending=[True, False])[
            [
                "segment_axis",
                "segment_value",
                "n_conversations",
                "rate_in_segment",
                "lift_vs_global",
                "pct_complaint_flag",
                "pct_regulatory_flag",
                "pct_required_intervention",
            ]
        ]
    )

print("Rule-first assets:")
print("- CSV: dev/segment_analysis_output/rule_segment_matrix.csv")
print("- Charts: dev/segment_analysis_output/charts/by_rule/")